# Candidate Pathogenic Repeat Identification

This notebook documents the scoring mechanism used to identify candidate pathogenic GCN repeats in 5'UTRs.

## Approach
We developed a composite pathogenicity score based on:
1. Population-level variability features from the repeat catalog
2. AlphaGenome-predicted functional consequences of repeat expansion

The score is computed by z-normalizing each discriminating feature relative to the non-pathogenic distribution and averaging across features.

## 1. Feature Selection

Features were selected based on Mann-Whitney U tests comparing the 7 known pathogenic repeats against 6,643 non-pathogenic repeats, with FDR correction.

In [ ]:
import os, pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = '/blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/data'
EXTRACTED_DIR = '/blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds/data/extracted'
OUT_DIR = '/blue/zhou/leizhou/Agents/Biomni_Rpts_Ds/Rpt_Ds'
print('Setup complete')


In [ ]:
pathogenic_df = pd.read_csv(DATA_DIR + '/B_5UTR_Pathogenic_GCNs.txt', sep='\t')
pathogenic_set = set(pathogenic_df['LocusId'].tolist())
catalog_df = pd.read_csv(DATA_DIR + '/B_Cat_5UTR_GCNs_masked.tsv', sep='\t')
catalog_df['is_pathogenic'] = catalog_df['LocusId'].isin(pathogenic_set)

def load_stats_file(filepath, expansion, data_type):
    df = pd.read_csv(filepath, sep='\t')
    df['expansion'] = expansion
    df['data_type'] = data_type
    return df

stats_dfs = []
for expansion in ['2x', '5x', '20x']:
    for data_type in ['exp', 'epi', 'splice']:
        fp = EXTRACTED_DIR + '/all_variants_%s_%s_stats.tsv' % (expansion, data_type)
        df = load_stats_file(fp, expansion, data_type)
        stats_dfs.append(df)

all_stats = pd.concat(stats_dfs, ignore_index=True)

feature_rows = {}
for _, row in all_stats.iterrows():
    vid = row['variant_id']
    if vid not in feature_rows:
        feature_rows[vid] = {}
    prefix = row['output_type'] + '_' + row['expansion']
    feature_rows[vid][prefix + '_mean_q'] = row['mean_q']
    feature_rows[vid][prefix + '_mean_raw'] = row['mean_raw']
    feature_rows[vid][prefix + '_max_abs_raw'] = row['max_abs_raw']
    feature_rows[vid][prefix + '_frac_neg_q'] = row['frac_neg_q']

wide_df = pd.DataFrame.from_dict(feature_rows, orient='index')
wide_df.index.name = 'variant_id'
wide_df = wide_df.reset_index()
wide_df['is_pathogenic'] = wide_df['variant_id'].isin(pathogenic_set)

cat_cols = ['LocusId', 'CanonicalMotif', 'NumRepeatsInReference', 'ReferenceRepeatPurity',
            'LPSLengthStdevFromHPRC100', 'StdevFromIllumina174k', 'StdevFromT2TAssemblies',
            'FlanksAndLocusMappability', 'GencodeGeneName', 'GencodeGeneId',
            'GencodeGeneRegion', 'GencodeTranscriptId', 'is_pathogenic']
full_df = wide_df.merge(catalog_df[cat_cols], left_on='variant_id', right_on='LocusId',
                         how='left', suffixes=('', '_cat'))
full_df = full_df.drop(columns=['LocusId', 'is_pathogenic_cat'])
print('Feature matrix:', full_df.shape)


In [ ]:
# Run statistical tests to identify discriminating features
exclude_cols = ['variant_id', 'is_pathogenic', 'CanonicalMotif', 'GencodeGeneName',
                'GencodeGeneId', 'GencodeGeneRegion', 'GencodeTranscriptId']
feature_cols = [c for c in full_df.columns if c not in exclude_cols]

path_full = full_df[full_df['is_pathogenic']]
nonpath_full = full_df[~full_df['is_pathogenic']]

results = []
for col in feature_cols:
    pv = path_full[col].dropna().values
    nv = nonpath_full[col].dropna().values
    if len(pv) < 3 or len(nv) < 3:
        continue
    try:
        stat, pval = mannwhitneyu(pv, nv, alternative='two-sided')
        pooled_std = np.sqrt((np.std(pv)**2 + np.std(nv)**2) / 2)
        d = (np.mean(pv) - np.mean(nv)) / (pooled_std + 1e-10)
        results.append({'feature': col, 'path_mean': np.mean(pv),
                        'nonpath_mean': np.mean(nv), 'cohens_d': d, 'p_value': pval})
    except:
        pass

results_df = pd.DataFrame(results)
_, q_vals, _, _ = multipletests(results_df['p_value'], method='fdr_bh')
results_df['q_value'] = q_vals
results_df = results_df.sort_values('p_value')

print('Top discriminating features (FDR < 0.10):')
sig_features = results_df[results_df['q_value'] < 0.10]
print(sig_features[['feature', 'path_mean', 'nonpath_mean', 'cohens_d', 'p_value', 'q_value']].to_string())


## 2. Score Computation

In [ ]:
# Composite score: z-normalize each feature relative to non-pathogenic distribution
# Direction: +1 if pathogenic > non-pathogenic, -1 otherwise

scoring_features = ['LPSLengthStdevFromHPRC100', 'NumRepeatsInReference', 'StdevFromT2TAssemblies',
    'CAGE_2x_mean_q', 'CAGE_5x_mean_q', 'CAGE_20x_mean_q',
    'CAGE_2x_mean_raw', 'CAGE_5x_mean_raw', 'CAGE_20x_mean_raw',
    'CAGE_2x_frac_neg_q', 'CAGE_5x_frac_neg_q', 'CAGE_20x_frac_neg_q',
    'RNA_SEQ_5x_max_abs_raw', 'RNA_SEQ_20x_max_abs_raw',
    'RNA_SEQ_5x_mean_raw', 'RNA_SEQ_20x_mean_raw',
    'ATAC_2x_mean_q', 'ATAC_5x_mean_q', 'ATAC_20x_mean_q',
    'ATAC_2x_frac_neg_q', 'ATAC_5x_frac_neg_q', 'ATAC_20x_frac_neg_q',
    'DNASE_2x_mean_q', 'DNASE_5x_mean_q',
    'DNASE_2x_frac_neg_q', 'DNASE_5x_frac_neg_q', 'DNASE_20x_frac_neg_q',
    'CHIP_TF_2x_mean_q', 'CHIP_TF_2x_frac_neg_q',
    'CHIP_TF_5x_frac_neg_q', 'CHIP_TF_20x_frac_neg_q',
    'CHIP_HISTONE_20x_mean_q', 'CHIP_HISTONE_20x_frac_neg_q']
scoring_features = [f for f in scoring_features if f in full_df.columns]

print('Scoring features used: %d' % len(scoring_features))
print()

# Determine direction from statistical results
direction_map = {}
for feat in scoring_features:
    row = results_df[results_df['feature'] == feat]
    if len(row) > 0:
        direction_map[feat] = 1 if row['path_mean'].values[0] > row['nonpath_mean'].values[0] else -1
    else:
        direction_map[feat] = 1

print('Feature directions (+ = pathogenic has higher values):')
for feat in scoring_features[:10]:
    print('  %s: %+d' % (feat, direction_map[feat]))
print('  ...')

# Compute reference stats from non-pathogenic
ref_stats = {}
for feat in scoring_features:
    vals = full_df[~full_df['is_pathogenic']][feat].dropna().values
    ref_stats[feat] = {'mean': np.mean(vals), 'std': np.std(vals) + 1e-10}

# Compute z-scores and composite score
z_list = []
for feat in scoring_features:
    z = direction_map[feat] * (full_df[feat] - ref_stats[feat]['mean']) / ref_stats[feat]['std']
    z_list.append(z.values)
z_matrix = np.column_stack(z_list)
full_df['composite_score'] = np.nanmean(z_matrix, axis=1)

all_scores = full_df['composite_score'].values
full_df['percentile_rank'] = [np.mean(all_scores < s) * 100 for s in all_scores]

print()
print('Score formula: composite_score = mean(direction_i * (feature_i - ref_mean_i) / ref_std_i)')
print('where ref_mean and ref_std are computed from non-pathogenic variants')


## 3. Validation: Known Pathogenic Repeats

In [ ]:
# Validate: all known pathogenic repeats should score high
print('=== VALIDATION: Known Pathogenic Repeats ===')
path_res = full_df[full_df['is_pathogenic']].sort_values('composite_score', ascending=False)
print(path_res[['variant_id', 'GencodeGeneName', 'NumRepeatsInReference',
                'LPSLengthStdevFromHPRC100', 'composite_score', 'percentile_rank']].to_string())

print()
print('All pathogenic repeats above 92nd percentile: %s' % 
      str(all(path_res['percentile_rank'] > 90)))
print('Min pathogenic percentile: %.1f%%' % path_res['percentile_rank'].min())
print('Mean pathogenic percentile: %.1f%%' % path_res['percentile_rank'].mean())

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
np_scores = full_df[~full_df['is_pathogenic']]['composite_score']
p_scores = full_df[full_df['is_pathogenic']]['composite_score']
p_genes = full_df[full_df['is_pathogenic']]['GencodeGeneName'].values

ax.hist(np_scores, bins=80, alpha=0.7, color='steelblue', label='Non-pathogenic (n=6643)', density=True)
for i, (v, g) in enumerate(zip(p_scores, p_genes)):
    ax.axvline(v, color='red', alpha=0.7, linewidth=2, label=g if i == 0 else '')
    ax.text(v, ax.get_ylim()[1] * 0.9 if ax.get_ylim()[1] > 0 else 0.5, g,
            rotation=90, fontsize=7, ha='right', color='darkred')
ax.axvline(p_scores.mean(), color='darkred', linewidth=2.5, linestyle='--',
           label='Pathogenic mean=%.2f' % p_scores.mean())
ax.set_xlabel('Composite Pathogenicity Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Composite Score: Known Pathogenic Repeats vs Non-Pathogenic', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR + '/validation_score.png', dpi=150, bbox_inches='tight')
plt.show()
print('Validation figure saved.')


## 4. Top 50 Candidate Identification

In [ ]:
# Identify top 50 candidates
top50 = full_df[~full_df['is_pathogenic']].sort_values('composite_score', ascending=False).head(50).copy()
top50.insert(0, 'rank', range(1, 51))

# Add key features
key_ag_cols = ['CAGE_2x_mean_q', 'CAGE_5x_mean_q', 'CAGE_20x_mean_q',
               'ATAC_2x_mean_q', 'ATAC_5x_mean_q',
               'CHIP_TF_2x_mean_q', 'RNA_SEQ_5x_max_abs_raw']
key_ag_cols = [c for c in key_ag_cols if c in top50.columns]

output_cols = ['rank', 'variant_id', 'GencodeGeneName', 'GencodeGeneId', 'GencodeGeneRegion',
               'CanonicalMotif', 'NumRepeatsInReference', 'LPSLengthStdevFromHPRC100',
               'StdevFromT2TAssemblies', 'composite_score', 'percentile_rank'] + key_ag_cols
output_cols = [c for c in output_cols if c in top50.columns]

top50_out = top50[output_cols]
top50_out.to_csv(OUT_DIR + '/Top_Candidate_Pathogenic_repeats.csv', index=False)
print('Top 50 candidates saved to Top_Candidate_Pathogenic_repeats.csv')
print()
print(top50_out[['rank', 'variant_id', 'GencodeGeneName', 'NumRepeatsInReference',
                  'LPSLengthStdevFromHPRC100', 'composite_score', 'percentile_rank']].to_string())


## 5. Mechanistic Hypotheses for Top Candidates

In [ ]:
# Mechanistic hypotheses for top candidates
print('=== MECHANISTIC HYPOTHESES FOR TOP CANDIDATES ===')
print()

top10 = full_df[~full_df['is_pathogenic']].sort_values('composite_score', ascending=False).head(10)

for _, row in top10.iterrows():
    gene = row['GencodeGeneName']
    vid = row['variant_id']
    score = row['composite_score']
    lps_stdev = row.get('LPSLengthStdevFromHPRC100', float('nan'))
    n_repeats = row.get('NumRepeatsInReference', float('nan'))
    cage_20x = row.get('CAGE_20x_mean_q', float('nan'))
    atac_5x = row.get('ATAC_5x_mean_q', float('nan'))
    
    print('Gene: %s (%s)' % (gene, vid))
    print('  Score: %.3f (%.1f percentile)' % (score, row['percentile_rank']))
    print('  Ref repeats: %s, LPS stdev: %s' % (str(n_repeats), str(round(lps_stdev, 3) if not np.isnan(lps_stdev) else 'NA')))
    print('  CAGE 20x mean_q: %.4f, ATAC 5x mean_q: %.4f' % (cage_20x, atac_5x))
    print('  Mechanism: GCN expansion in 5-UTR predicted to silence gene expression')
    print('             via chromatin compaction and transcription factor displacement')
    print()


In [ ]:
# Summary visualization: top 50 candidates
top50_plot = full_df[~full_df['is_pathogenic']].sort_values('composite_score', ascending=False).head(50)
path_plot = full_df[full_df['is_pathogenic']]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: scatter plot of LPS stdev vs composite score
ax = axes[0]
ax.scatter(full_df[~full_df['is_pathogenic']]['LPSLengthStdevFromHPRC100'].fillna(0),
           full_df[~full_df['is_pathogenic']]['composite_score'],
           alpha=0.2, color='steelblue', s=15, label='Non-pathogenic')
ax.scatter(top50_plot['LPSLengthStdevFromHPRC100'].fillna(0),
           top50_plot['composite_score'],
           alpha=0.7, color='orange', s=40, label='Top 50 candidates')
ax.scatter(path_plot['LPSLengthStdevFromHPRC100'].fillna(0),
           path_plot['composite_score'],
           color='red', s=100, zorder=5, label='Known pathogenic')
for _, row in path_plot.iterrows():
    ax.annotate(row['GencodeGeneName'],
                (row['LPSLengthStdevFromHPRC100'] if not np.isnan(row['LPSLengthStdevFromHPRC100']) else 0,
                 row['composite_score']),
                fontsize=8, xytext=(3, 3), textcoords='offset points', color='darkred')
ax.set_xlabel('LPS Length Stdev (HPRC100)', fontsize=11)
ax.set_ylabel('Composite Pathogenicity Score', fontsize=11)
ax.set_title('Population Variability vs Pathogenicity Score', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# Right: bar chart of top 20 candidates
ax = axes[1]
top20 = full_df[~full_df['is_pathogenic']].sort_values('composite_score', ascending=False).head(20)
colors_bar = ['orange'] * 20
ax.barh(range(20), top20['composite_score'].values, color=colors_bar, alpha=0.8)
ax.set_yticks(range(20))
ax.set_yticklabels(top20['GencodeGeneName'].values, fontsize=9)
ax.axvline(full_df[full_df['is_pathogenic']]['composite_score'].min(), 
           color='red', linestyle='--', linewidth=2, label='Min pathogenic score')
ax.set_xlabel('Composite Pathogenicity Score', fontsize=11)
ax.set_title('Top 20 Candidate Pathogenic Repeats', fontsize=11, fontweight='bold')
ax.invert_yaxis()
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR + '/candidate_identification.png', dpi=150, bbox_inches='tight')
plt.show()
print('Candidate identification figure saved.')
